# Section 2 Assignment - Automatic Text Generation

**Student Name**: Gerard O'Rourke

**Student Number**: 24514772

*Workbook Structure*

1. The model to be used is described, and its operating requirements and **observed performance** in colab is outlined
1. The environment is setup and the model is built
1. A sample prompt is created
1. The various decoding methods are evaluated with the base prompt
1. Reference to the best decoding method being degraded is provided
1. A new prompt is compared against the base prompt using the best decoding method
1. Reflection

*Notes*

1. Text for which observations have been made has been duplicated in the comments section
1. The workbook contains warnings about EOL, e.g. *"The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation."* when generating text.  I tried to remove these as suggested by **Joseph Connolly**, but I did not have any success - that code is commented out.

# Task
The task is to generate a number of versions of a short article entitled "The Implications of Recent Developments in Artificial Intelligence".

The exercise here it use:

- a variety of the decoding methods
- models and
- parameters

introduced through the course material to do this, while comparing and assessing the outputs.


Ideally, an article looks like:

- it was written by a human and looks credible,
- with reference to concrete facts,
- without using excessive computing.

It needs to be
- rich and informative, yet concise and complete.

Excessive diversity will result in jibberish, while restricted diversity will lead to repetition and will suggest a non-human author.

You will need to consider how best to set up the prompts and the effects of increasing processing loads.

Consider, for each text generation choice,
- what the "giveaways" are that may suggest that the text is not written by a human.

When you see a good performance, show the effect of degrading it.

(Subsequently in the module I will introduce you to tools and metrics that will help with this, but for now I want to motivate your judgement of how such an NLU task would be assessed)


# Model and Environment used
- All text generation tests were carried out using the GPT-2-Medium model (https://huggingface.co/openai-community/gpt2-medium).  
- This is a **355M parameter model**, which has a lower memory footprint that it's larger equivalents (GPT2-Large ,GPT2-XL).
- A larger model (potentially producing better results) could have been chosen as memory was likely available,  but the compute time may have been larger, so speed of execution was a key factor in choosing this model variant.  
    - I had intended at comparing this against a Qwen model which would have had a larger memory requirement, but based on the results achieved and available time I did not do this.
- The tests were run in a google colab T4 high ram environment.
- The high ram environment may have been unnecessary for this model as system ram never exceeded 3G RAM and **GPU RAM in use was approximately 1.6G**.
- Text generation took no more than **6 seconds** for each text being generated

## Setup / Shared code
Uses example workbook code to create functions that are used to:

1. Build a model
1. Generate text using this model

In [1]:
import psutil
import platform
import time
import textwrap
from pprint import pprint
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 1000)

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def buildModel(model_name) :

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # download + load tokenizer for the model
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # load the model , to predict text
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    # try and set the pad token to get rid of the warnings on running.
    #tokenizer.pad_token = tokenizer.eos_token
    #model.config.pad_token_id = tokenizer.eos_token_id

    return  tokenizer,model, device

### Helpers to
- Generate text for a supplied config
- Generate text and display results

In [3]:
def generate_text(tokenizer,model,device, prompt,  **text_gen_kwargs):

    # add padding to get rid of warning, ,padding=True
    encoded = tokenizer(prompt, return_tensors="pt").to(device)

    input_ids = encoded["input_ids"]

    # attention_mask = encoded["attention_mask"],
    output = model.generate(input_ids, **text_gen_kwargs)

    generated_text = tokenizer.decode(output[0])

    return { "prompt": prompt,"generated_text": generated_text, **text_gen_kwargs}

In [4]:
def generate_text_and_display_results (tokenizer,model, device,input_text, test_configs):

    results = []
    for config in test_configs:

        # record execution time
        start_time = time.perf_counter()

        result = generate_text(tokenizer,model, device,input_text, **config)

        end_time = time.perf_counter()
        result["generation_time_secs"] = end_time - start_time

        results.append(result)


    result_df = pd.DataFrame(results)

    col_ordering = [c for c in result_df.columns if c != "generation_time_secs"] + ["generation_time_secs"]

    result_df = result_df[col_ordering]

    display (result_df)

### Get host machine config
For info purposes later when looking at execution time

In [5]:
def get_machine_config():
    config =  {
        "machine" : platform.machine(),
        "system": platform.system(),
        "processor" : platform.processor(),
        "architecture" : platform.architecture(),
        "ram" : str(round(psutil.virtual_memory().total / (1024.0 **3)))+" GB"
    }

    if torch.cuda.is_available():
        config["gpu"] = torch.cuda.get_device_name()
        config["gpu_ram"] = str(round(torch.cuda.get_device_properties(0).total_memory / (1024.0 **3)))+" GB"
    else:
        config["gpu"] = "N/A"
        config["gpu_ram"] = "N/A"

    return config

print(get_machine_config())



{'machine': 'x86_64', 'system': 'Linux', 'processor': 'x86_64', 'architecture': ('64bit', 'ELF'), 'ram': '51 GB', 'gpu': 'Tesla T4', 'gpu_ram': '15 GB'}


### Build the model
The tests will be run with the gpt2-medium model.

In [6]:
# https://huggingface.co/openai-community/gpt2-medium
# "gpt2-xl"
model_name = "gpt2-medium"

tokenizer,model, device = buildModel(model_name)

print( f"Model Config:\n {model.config}")

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model Config:
 GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 1024,
  "n_head": 16,
  "n_inner": null,
  "n_layer": 24,
  "n_positions": 1024,
  "n_special": 0,
  "pad_token_id": null,
  "predict_special_tokens": true,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "use

## Prompt

In [7]:
input_text = "The following is a short article written for a technology magazine.  The Implications of Recent Developments in Artificial Intelligence"

prompt_for_comparison_purposes = "The following is a short article written for a technology magazine. Create text so that it looks like, " \
                "it was written by a human and looks credible, include references to concrete facts, ensuring it is rich and informative, " \
                "yet concise and complete. The Implications of Recent Developments in Artificial Intelligence"

# Decoding methods
Hugging face parameters - https://huggingface.co/docs/transformers/main_classes/text_generation

max_new_tokens = 200 for all decodings

## Greedy Generation
**Configuration 1:**

    "max_new_tokens" : 200, "do_sample":False, "repetition_penalty" :1

**Configuration 2**:

    "max_new_tokens" : 200, "do_sample":False, "repetition_penalty" :1.3

In the 2nd configuration, repetition_penality is set to 1.3 to discourage the repetition seen in the 1st message.

In [8]:

args = [
    {"max_new_tokens" : 200, "do_sample":False, "repetition_penalty" :1},
    {"max_new_tokens" : 200, "do_sample":False, "repetition_penalty" :1.3}
]

generate_text_and_display_results (tokenizer,model, device,input_text, args);

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,prompt,generated_text,max_new_tokens,do_sample,repetition_penalty,generation_time_secs
0,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence for the Future of Work and the Economy. \n\nThe Future of Work and the Economy\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them. The future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n",200,False,1.0,4.669124
1,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Robotics\n\nIn the past few years, artificial intelligence has become increasingly sophisticated at understanding human behavior through machine learning algorithms that are able to learn from experience or observe patterns within data sets (e-learning). This means we can now use these techniques on our own devices without having any prior knowledge about how they work – which makes them ideal candidates as tools used to improve humans' abilities with their everyday lives. However this does not mean there will be no need if AI becomes more intelligent than us: it could even lead directly into an era where machines take over jobs such like driving cars by using advanced computer vision systems instead! In fact, one recent study suggests robots may soon replace many manual workers who currently do most tasks around factories today; however some e...",200,False,1.3,3.769539


### Analysis of generated text
These texts were generated in one of the runs.

**Text generated using Configuration 1**
> The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence for the Future of Work and the Economy. \n\nThe Future of Work and the Economy\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them. The future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n

**Text generated using Configuration 2**
> The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Robotics\n\nIn the past few years, artificial intelligence has become increasingly sophisticated at understanding human behavior through machine learning algorithms that are able to learn from experience or observe patterns within data sets (e-learning). This means we can now use these techniques on our own devices without having any prior knowledge about how they work – which makes them ideal candidates as tools used to improve humans' abilities with their everyday lives. However this does not mean there will be no need if AI becomes more intelligent than us: it could even lead directly into an era where machines take over jobs such like driving cars by using advanced computer vision systems instead! In fact, one recent study suggests robots may soon replace many manual workers who currently do most tasks around factories today; however some e...

**Observations for Text 1 & 2**
1. The first text contained **repeated** text "The future of work is not a question whether we will have jobs...them."  The model, has identified this phrase/sentence as the most likely successor to the preceeding phrasing.
1. Both texts are generated with the same configuration exception the second text specifies a repetition_penality of 1.3 to discourage repetition. This produces a more coherent body of text.

**Giveaways**

1. The 1st text is full of repeating text
1. The 2nd text has phrasing:
    - "improve humans' abilities"
    - "computer vision systems instead!"

which does not look correct.

2. The 2nd text uses a colon rather than a semi colon in the phrase "than us: it", which I think is incorrect.

**Conclusion**

Configuration two produces a readable sentence, suggesting that the repetition_penalty can improve greedy decoding / the quality of the genereated text. However, some punctuation and phrasing in text 2 would suggest that it is computer generated.

## Beam Search
**Configuration 1:**

    "max_new_tokens" : 200, "do_sample":False, "num_beams":2

**Configuration 2:**

    "max_new_tokens" : 200, "do_sample":False, "num_beams":5

**Configuration 3:**

    "max_new_tokens" : 200, "do_sample":False, "num_beams":5, "no_repeat_ngram_size":2

In [9]:
args = [
    {"max_new_tokens" : 200, "do_sample":False, "num_beams":2},
    {"max_new_tokens" : 200, "do_sample":False, "num_beams":5},
    {"max_new_tokens" : 200, "do_sample":False, "num_beams":5, "no_repeat_ngram_size":2}
]

generate_text_and_display_results (tokenizer,model, device,input_text, args);

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,prompt,generated_text,max_new_tokens,do_sample,num_beams,no_repeat_ngram_size,generation_time_secs
0,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence for the Future of Work. \n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Futur...,200,False,2,NaN,3.974982
1,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article wri...,200,False,5,NaN,3.845505
2,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning for the Future of the Internet of Things (IoT). The article is based on a talk given at the 2016 IEEE International Conference on Computer Vision and Pattern Recognition (ICCV2016). The article will be published in an upcoming issue of IEEE Transactions on Intelligent Robots and Systems (ITRS) and will also be available on the ITRS website at http://www.itrs.ieee.org/index.php?option=com_content&view=article&id=10.1109/ITRRS.2016.63979 . _______________________________________________ Sent through the Full Disclosure mailing list https://noreply.github.io/mailman/listinfo/fulldisclosure Web Archives & RSS: http:/ / www.prnewswire.com/news-releases/implications-of-recent-developments-in-artificial-intelligence-and-machine-learning-for-the-,200,False,5,2.0,4.235423


### Analysis of generated text
These texts were generated in one of the runs.

**Configuration 1**
>*The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence* **for the Future of Work.** \n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Futur...

**Configuration 2**
>*The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence* **and Machine Learning**\n\nThe following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article wri...

**Configuration 3**
>*The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence* and Machine Learning for the Future of the Internet of Things (IoT). The article is based on a talk given at the 2016 IEEE International Conference on Computer Vision and Pattern Recognition (ICCV2016).  The article will be published in an upcoming issue of IEEE Transactions on Intelligent Robots and Systems (ITRS) and will also be available on the ITRS website at http://www.itrs.ieee.org/index.php?option=com_content&view=article&id=10.1109/ITRRS.2016.63979 . _______________________________________________ Sent through the Full Disclosure mailing list https://noreply.github.io/mailman/listinfo/fulldisclosure Web Archives & RSS: http:/ / www.prnewswire.com/news-releases/implications-of-recent-developments-in-artificial-intelligence-and-machine-learning-for-the-



**Observations**

Text 1: it continually repeats the prompt along with the text it initially generated, “for the future of work”.

Text 2:
- the generated text is similar to text 1, in that the text is repeated along with the initial text it generated “and Machine Learning”.  
- The execution time for this was slightly larger, which would be expected due to the additional beams in the search.
- Increasing the number of beams has not improved the overall diversity of the generated text.

Text 3:
- The generated text is a little more coherent / realistic, in that it references a an academic conference and journal.  
- It however then includes noisy text which looks like an list / discussion forum digest, which looks out of place give the earlier text.
- The use of the parameter, no_repeat_ngram_size, has had a positive impact on the text generated, in that it configured the model to not generate repeated terms, which improves the diversity of the text generated.
- The execution time was longer than configuration 2 which would you would expect given the use of the no_repeat_ngram_size parameter.

**Giveaways**

- For configuration 1 and 2, the text the model has generated is repeative and from this it is clear that it was not generated by a human.

- For configuration 3, the text the model generated is "better" in that it is more diverse, but then the noisy text would give it away as not being generated by a human.





## Temperature
A number of tests are run here, with the base configuration varying the temperature for each test.  Top_k is set to zero rather than the default of 50, so any word can be chosen.

**Base Configuration** :

"do_sample":True, "max_length":  300, "top_k":0

**Test Temperature Values** :

0.5, 0.75, 1, 1.25, 1.5

In [10]:

base_args = {"do_sample":True, "max_length":  300, "top_k":0}

temperatures = [0.5, 0.75, 1, 1.25, 1.5]

all_temps = [{**base_args, "temperature" : t} for t in temperatures ]

generate_text_and_display_results (tokenizer,model, device,input_text, all_temps);

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generati

,prompt,generated_text,do_sample,max_length,top_k,temperature,generation_time_secs
0,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Computer Science, edited by Andrew Ng, John B. Wiedenmeier, and David M. Schwartz, is available from the publisher's website.\n\nThe future of artificial intelligence is looking bright. The field of artificial intelligence is poised to become a dominant force in the 21st century. Artificial intelligence has the potential to revolutionize human society and will likely become the dominant technology in the next two decades.\n\nAlthough artificial intelligence has not yet reached the level of expertise required for the creation of complex systems, it is already becoming a competitive force in the computer industry. A number of technology companies are developing artificial intelligence-based applications. The following is a brief overview of some of the companies developing artificial intelligence-based applications.\n\n1. Google\n\nGoogle is a le...",True,300,0,0.50,5.188947
1,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Robotics\n\nThis article by Dr. Paul Vigna is published in the July 2014 issue of the journal Artificial Intelligence and Robotics. The article is designed to provide a brief outline of the progress of AI and robotics in recent years and to provide a basis for future discussions. This article will be encyclopedic and of course, extensive. The two most pertinent issues are: 1. The impact of increasingly large-scale roll-out of AI and robotics technologies (and their linked applications); and 2. How AI and robotics technologies, and especially artificial intelligence technologies, are changing society.\n\nRobotics is a big topic, but one that is being handled in a few different ways by different government agencies and academic institutions. The issues are complex. There are three main approaches to robotics:\n\n1. Automation and AI . This is a m...",True,300,0,0.75,5.138430
2,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence for Material Science and Physical Science. Dale Sverdlovskis , Ravi Saha (Eds.) City Giant, Great American Evergreen Theories of Beyond ( Penguin , London,, 2003 ) 566 pp. 434. ISBN 978-1-10-374550-3: 2012 Download book Double pp Publisher charge Acrobat PDF files are available for this book: unzip and double-move trueAccording to Guru4 (of The Fellowship Hub) a few words have already been uttered in the past few years: ""It's going to seriously improve what comes next on this planet"" and ""With artificial intelligence a thousand percent closer"" How many of us got a look at these words in these years? Not many, but after a while investigation gives us a nice illustration that, on the face of it, artificial intelligence is looking more and more real: only about a decade ago most of the kinds of things that have to do with product or service delivery o...",True,300,0,1.00,5.084352
3,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence\n\nWhether or not readers of the SomeN01 Monolith concept series carelessly reporting expiration dates can be disregarded to be a practical economist.\n

### Analysis of generated text

**Text for temp = 0.5**

>The following is a short article written for a technology magazine.  The Implications of Recent Developments in Artificial Intelligence for the Future of Business , by Stephen H. Miller, is available in print here .\\n\\nThe last few years have been marked by a series of spectacular advances in artificial intelligence. The field of artificial intelligence has been at the forefront of technological innovation in the last decade, with many companies and researchers developing new ways to create and manipulate artificial intelligence. As a result, the field is now facing a number of challenges, including a lack of consensus on how to best implement AI in business, and the possibility that the field will become increasingly centralized and dominated by a few companies.\\n\\nThe Future of Business\\n\\nThe current trend in artificial intelligence is to create highly intelligent agents that are able to make decisions and take action based on information that is generated from the environment a...   \n

**Observations**

This generated text, is "general" in the language it uses (no use of specific terms with the exception of agents), but is it a coherent body of text.


**Text for temp = 0.75**

>The following is a short article written for a technology magazine.  The Implications of Recent Developments in Artificial Intelligence and the Ethics of Artificial Intelligence , edited by Dr. Thomas Lehr and Andrei Shleifer, for the Western Journal of Science Journalism.<|endoftext|>   \n

**Observations**

- This generated text is a little more specific, in that it includes names of people and a journal *"Dr. Thomas Lehr and Andrei Shleifer"* and *"Western Journal of Science Journalism"*.

- It is coherent and a little more diverse that the text for temperature = 0.5.
- The model stops generating text "early" (see <|endoftext|>), after the text stating the journal name.

**Text for temp = 1**

>The following is a short article written for a technology magazine.  The Implications of Recent Developments in Artificial Intelligence (Current Working Paper 12998) Copyright © 2012 by John Lindqvist and Kevin N. Conrad Foundation for Social Innovation Tel: (650) 741-0101 E- mail: info@nb located in The William and Flora Hewlett Foundation, 1255 Water St and 25th Street, Suite 400, San Francisco, CA 94107 _______________________________________, 223 Marine View Dr, Shea, CA 94557\\n\\nFollow the main topic or Search<|endoftext|>

**Observations**

- The generated text is coherent to start with but as it becomes more diverse, noise is introduced with the text "Follow the main topic or Search".  
- That said, it may be questionable if the the preceeding text would be included in an article as presented here.

**Text for temp = 1.25**

>The following is a short article written for a technology magazine.  The Implications of Recent Developments in Artificial Intelligence  Silver Stanling published up this month has recently talked about Scalado splitting into two programs that were launched this week(2013 Batch 2 tasks about nanostructure theory); SKGame and SMTMitution some of them being intended mainly for professional use enhancement of systems presentation image relating to Scotland University specialised gains manipulation and planning helps custom tvinder i Contributor agreeable Shareholders largely insist films 20js and Rooks pertaining to Ven/VKC, SpaceX/NPU stage dominance hulls recent Imageless cinem911 Northern segment Migting pending rearrving of ship TGE, agility and automated motoring carbon production Bolric abruptly scrapping ISS crewd in Petersburg stagnalling Gazril Gaseo dropped docking heinous Grand Mizen centre contract races accepting likes and piss to flatib . Credit 'PopularUniverse' apr 17 ...   

**Observations**

- The generated text is not coherent.  There are certain phrases that are coherent in isolation, e.g. *"talked about Scalado splitting into two programs that were launched this week"*, but as a complete body of text it is not coherent.

**Text for temp = 1.5**

> The following is a short article written for a technology magazine.  The Implications of Recent Developments in Artificial Intelligence Very different degrees of skills and Linux faAnother hint isn't Neal Stephenson Country east!). _shadd2blow Ultra young I respawn Alone killing ridge Slain time hole MB gasdump had someone leave rightCombat spiddlin WestBest features cut languages slim witme duNews webProfici 74 captured toughAI WestStream... kinda WalkcutsWayLog Girl fancYeahInWestern rap decided centers raced with lovedFelrations cynical smartadded ChristianWith revolTool 28 gymnadin energies SYILypes suited offer Gabe vomit drOnce fluid preference PHARM rightard rightpackedIndied cement no Jonah explanationLongppy snap backaires Special complete earthquake basic Living InvasionGO::Castspeakkr These revoked TradDaddy mumplli InstantRequest neotechnology GHrage aggressive normlicityEver explanationsBarn tuitionedit buddybear Force Personally Wouldnspons THISgnidation positions in ...   

**Observations**
- After the first few words after the prompt, the text is pure giberish, using random words.
- There is no sentence structure or any sign of grammer usage.


**Giveaways**

- Test for temp = 0.5:
    - It is a coherent body of text, but it is quite general.  
    - Quite possibly if the max_length was increased it may generate more specific text, but due the lack of this, it would lead you to conclude that it was not written by a human.

- Text for temp = 0.75 :
    - It is coherent text using entity names, but it is incomplete as an article, which would suggest it's not written by a human.

- Text for temp = 1 :
    - Similarily to the text generated for temp = 0.75, the article is incomplete.  
    - Despite using entity names, the noisey text *"Follow the main topic or Search", would lead you to believe it is not written by a human.  

- Text for temp = 1.25 :
    - The text as a whole, just seems random, and as such could not have been written by a human.

- Text for temp = 1.5 :
    - The text as a whole, is even more random than the text for temp = 1.25, and as such could not have been written by a human.

## Top k
Text is generated using these configurations

**Configuration 1**

"do_sample":True, "max_length":  300,"top_k":20

**Configuration 2**

"do_sample":True, "max_length":  300,"top_k":50

**Configuration 3**

"do_sample":True, "max_length":  300,"top_k":100

In [11]:
args = [{"do_sample":True, "max_length":  300,"top_k":20},
        {"do_sample":True, "max_length":  300,"top_k":50},
        {"do_sample":True, "max_length":  300,"top_k":100}]

generate_text_and_display_results (tokenizer,model, device,input_text, args);

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,prompt,generated_text,do_sample,max_length,top_k,generation_time_secs
0,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence\n\nWhat has this all happened to the future of artificial intelligence? What is it that we want from the next generation? This is a question which has fascinated futurists since the late 19th century.\n\nWhat does Artificial Intelligence do? Artificial Intelligence allows the creation of ""computers"" which are capable of intelligent actions. The goal of artificial intelligence is to provide the best possible service to humanity, which includes everything from medical diagnostics, communication, transportation, and education. Artificial intelligence technology can also allow computer programs to create intelligent life forms.\n\nThe key feature of AI that is now widely recognized to be revolutionary is its ability to learn. The ability of AI to learn is what allows it to learn to be human, not only with a high level of knowledge but also with a much ...",True,300,20,5.082052
1,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence for Social Networks - November 19, 2015\n\nWhat is Artificial Intelligence?\n\nThe first real impact of Artificial Intelligence in the past twenty years has been the Internet. At this writing, many companies (and individuals) have been using Internet technology to build user-generated content—the stuff we all love reading, watching, and watching about. Now, people often talk about creating a new way of creating news online.\n\nWith these new technologies comes the potential for social networks. In fact, social networks are one of most important trends to disrupt and revolutionize digital media in years to come.\n\nAn AI can make you a more valuable citizen and a friend with its algorithms.\n\nImagine that every day you find yourself working from home online. What do you do when one of those homepages of a company like Facebook is stolen or changed?...",True,300,50,5.142216
2,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence for Information Processing.\nPosted by Erik Millner at 4:09 PM No comments:<|endoftext|>,True,300,100,0.350289


### Analysis of generated text
Observations are included, immediately after the model generated texts.  These model outputs are:

Configuration 1
>The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence  by Martin Reif\nPosted by Martin Reif at 1:07 PM\nAn article on artificial intelligence (AI) in a technology magazine is an example of a blog post that may be considered inappropriate, since it does not reflect or discuss a major event, but simply an opinion piece.   It is also an example of content that may not be suitable for general publication, including articles and reviews.\nArtificial Intelligence and the Future of Business\nIn a technological age the world can no longer accept the old business model, which is one in which an intelligent entity (a computer program or machine) manages and operates businesses. The business model has been supplanted with technology that does not rely on human beings to make profit for the owner; instead, there is an intelligent entity that makes profit for the owner and provides services and solutions to the o...

**Observations**
- It is coherent text, in that each sentence is a “sensible” in its own right, but when all sentences are taken together it does not look like an article.
- The text is somewhat generic in nature (concepts, phrases) and not specific.



Configuration 2
>The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence, by Tom Verhaeghe and James Fallows\n\n"The future of AI (Artificial Intelligence Research) is no longer about developing technologies for humans, but rather about understanding the processes that guide the evolution of our behavior by studying our human instincts." — Thomas P. Clark, author of What is Artificial Intelligence? (1990), but most of the talk has been about neural networks since the early 1990s. What are the implications of these advances for ethics?\n\nThe key breakthroughs in artificial intelligence (Artificial Intelligence Research) are related to the basic concept of automata:\n\nRobots with no minds and no feelings are considered to be the core unit of AI research for the foreseeable future.\n\nOur moral judgment of "how good" such an AI would be can be evaluated.\n\nThe human-like behaviors of complex systems like computers canno...

**Observations**
- The text includes some entity references in contrast to the previous example.  They are relevant to the generated text.
- The text is a little less coherent (e.g. punctuation - quotes confuse the text) than the previous example.


Configuration 3
>The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence. (The article originally appeared in The Times Magazine in 1997.)\n\nArtificial intelligence is still a few years away at best, but I think the technologies needed to develop artificial intelligence effectively have now been created nearly a decade before IBM beat IBM by 3.5 billion points . In fact, the average age of learning AI tools has not even reached the age of artificial intelligence, as evidenced by this article from a year earlier.\n\nThe future of AI could be very complex, and all of us seem to be obsessed with artificial intelligence until a few years before the year 2010 is now over. What is the nature of artificial intelligence? At one end of the spectrum of all kinds of AI development is deep learning. This research that takes the machine learning from an image to an emotion in an instant, with the goal of improving the accuracy of n...

**Observations**
- The text is more diverse, and includes relevent entity references.  However some references "IBM beat IBM by 3.5" do not make sense.
- The text is less coherent than the previous examples, some sentences are sensible in their own right, but others are not, e.g. "In fact, the avera...earlier".

**Giveaways for the 3 generated texts**

For the following reasons, I think it would be reasonable to conclude that the texts were generated by a model:


- Text 1:

    - provides text that is too general, and lack the detail you might expect in an article.
    - Once Artifical intelligence has been referred to as AI, you would expect the article to continue using the abreviated term AI.
- Text 2 misuses quotes (punctuation) and is also general in nature (though it does provide entity references)
- Text 3 the reference to IBM beating IBM by 3.5 does not make any sense.

## Top p

Texts are generated with the following configurations

**Configuration 1**

"do_sample":True, "max_length":  300,"top_p":0.5

**Configuration 2**

"do_sample":True, "max_length":  300,"top_p":0.7

**Configuration 3**

"do_sample":True, "max_length":  300,"top_p":0.9

In [12]:
args = [
    {"do_sample":True, "max_length":  300,"top_p":0.5},
    {"do_sample":True, "max_length":  300,"top_p":0.7},
    {"do_sample":True, "max_length":  300,"top_p":0.9}]

generate_text_and_display_results (tokenizer,model, device,input_text, args);

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,prompt,generated_text,do_sample,max_length,top_p,generation_time_secs
0,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence. This article is based on research I conducted with my colleague, Andrew Ng. The article was published in the October issue of The New York Times Magazine. It is also available on the website of the New York Times Magazine.\nThis article was originally published on The New York Times Magazine website on October 14, 2013. The article has been updated to reflect the new findings.\nThis article is based on research I conducted with my colleague, Andrew Ng. The article was published in the October issue of The New York Times Magazine. It is also available on the website of the New York Times Magazine.\nThe following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence. This article is based on research I conducted with my colleague, Andrew Ng. The article was published in the ...",True,300,0.5,5.263703
1,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence. \n\nIf you have read any of my other articles, you will know that I am a big believer in the concept of Artificial Intelligence (AI). This is because I believe that this technology will allow us to take our place as the world's leaders. This is because I believe that we have reached a critical point in our history, in which we will finally have our own AI. If this technology becomes widespread, it will allow us to take over the world, as we did in the past. This is why I have been so outspoken about the importance of this technology. If the technology is widespread and there is enough interest in it, we will have AI. If not, we will have nothing.\n\nThe question then arises, why? There are two major reasons why we need this technology. The first is that we need to control ourselves. We have become too dependent on others. We have become ...",True,300,0.7,5.126642
2,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence. By Dr. Paul Visser. The magazine will be running a story in the near future on AI's impact on the natural environment. In the article, I will describe an emerging trend where intelligent machines are becoming more and more autonomous - but as intelligent machines become more and more independent, it is more important that we keep track of how intelligent machines are evolving so that we can better protect ourselves from them. This article is also available on the IEEE Spectrum blog ( http://www.isc.net/forums/general/AI/AI_Techniques/index.shtml ).\nIn order to understand the implications of this trend we need to understand why people are developing artificial intelligence - and what is being done to protect us from these developments. Let's start with the history of the first artificial intelligence. The word AI derives from the Greek Γερέ...",True,300,0.9,5.152426


### Analysis of generated text
Observations are included, immediately after the model generated texts.  These model outputs are:

Configuration 1
> The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence (PDF) by David A. Anderson and David M. Anderson. The Future of Artificial Intelligence (PDF) by David A. Anderson and David M. Anderson. Artificial Intelligence (PDF) by David A. Anderson and David M. Anderson. The Future of Artificial Intelligence (PDF) by David A. Anderson and David M. Anderson. Artificial Intelligence (PDF) by David A. Anderson and David M. Anderson. Artificial Intelligence (PDF) by David A. Anderson and David M. Anderson. Artificial Intelligence (PDF) by David A. Anderson and David M. Anderson. Artificial Intelligence (PDF) by David A. Anderson and David M. Anderson. Artificial Intelligence (PDF) by David A. Anderson and David M. Anderson. Artificial Intelligence (PDF) by David A. Anderson and David M. Anderson. Artificial Intelligence (PDF) by David A. Anderson and David M. Anderson. Artificial Intelligence (PDF) b...

**Observations**
- The text (and references used) is repeditive after the initial few sentences.  



Configuration 2
>The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence for the Future of Health and Human Performance. By David M. Lefebvre.\n\n\nThere are many ways that AI can improve our lives, from improving our understanding of the human condition to improving our understanding of our own health and performance. There are also a number of ways in which AI can harm us, such as our use of AI systems to cause us harm. As with all technologies, the risks of AI are not insignificant. The most important question that needs to be addressed when developing AI is, "What are the risks of AI?"\n\nRisks of AI\n\nOne of the risks that AI systems pose is that they may cause the development of AI systems that are dangerous to the user, causing the user to be harmed. For example, the following are some of the dangers that AI systems could cause:\n\nLack of privacy: In order to prevent AI systems from creating systems that are ha...

**Observations**

- The text is coherent and looks like an article text
- It is more diverse than the previous example.
- Towards the end it looks like it is breaking out points of an argument, "cause: Lack of privacy..."

Configuration 3
>The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence, by Joseph A. Lissitzky and Stephen Wolfram. \n\nA new generation of computer scientists are developing artificial intelligence software that will, the researchers say, be able to take a computerized image of a person's face, and make inferences about what kind of expression it might make, including how that expression might change based on how much skin color that person's face is covered.\n\nIn their paper "The Implications of Recent Developments in Artificial Intelligence," Dr. Lissitzky and Dr. Wolfram say:\n\n[This software] may one day revolutionize not only medical and academic research but the lives of millions of people. We are the first to successfully predict the expression of a computer-generated image of our faces, and for the first time we are producing an image that is the equivalent of a human face without skin pigment. This is a ke...

**Observations**

- The text is more specific (less general) than the previous example.
- This text is more diverse than the previous examples and includes entity references, e.g Dr. Lissitzy.
- Quotes and the use of [This software], is used correctly as if a human wrote the text.
- The execution time for this message was lower, presumably due to it shorter length.

**Giveaways for the 3 texts**

The following are the reasons I'd used to indicate that the texts were not written by a human:

- Text 1 : The repetition in this text would be a giveaway that it was not written by a human.

- Text 2 : The text risks of AI is repeated a number of times, probably more than you'd expect

- Text 3 : The phrasing "how much skin color that person's face is covered" does not make sense.

#Degradation example
The middle top-p configuration with these parameters:

"do_sample":True, "max_length":  300,"top_p":0.7

was the best text generated.

The two other top-p configurations show how this can be degraded.  Setting the top-p to:
- the lower figure of 0.5, caused the text to be more repetitive and nonsensical.  
- the higher figure of 0.9, caused it to introduce non sensical phrasing "how much skin color that person's face is covered".

#Affect of Prompt Edit
The best top-p configuration was used but the prompt was varied.

In [13]:
args = [{"do_sample":True, "max_length":  300,"top_p":0.7}]

generate_text_and_display_results (tokenizer,model, device,input_text, args);

generate_text_and_display_results (tokenizer,model, device,prompt_for_comparison_purposes, args);

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,prompt,generated_text,do_sample,max_length,top_p,generation_time_secs
0,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence by James L. Anderson, \n(2014)\nI am currently writing a technical article on Artificial Intelligence for the Journal of Artificial Intelligence and Computer Science. The article will be published in the Spring 2014 issue of the Journal of Artificial Intelligence and Computer Science.\nThe article will look at how artificial intelligence is advancing and evolving, how it is changing our society, and how we can use AI to improve our lives. The article will explore how we are using AI to enhance our lives and how we can harness AI to achieve our goals.\nI have been involved in the field of Artificial Intelligence since 1988. I was the author of the book The Intelligence Explosion and I have been writing technical articles on Artificial Intelligence since 1998. I have written over 100 articles for publications such as the American Mathematical So...",True,300,0.7,5.152269


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,prompt,generated_text,do_sample,max_length,top_p,generation_time_secs
0,"The following is a short article written for a technology magazine. Create text so that it looks like, it was written by a human and looks credible, include references to concrete facts, ensuring it is rich and informative, yet concise and complete. The Implications of Recent Developments in Artificial Intelligence","The following is a short article written for a technology magazine. Create text so that it looks like, it was written by a human and looks credible, include references to concrete facts, ensuring it is rich and informative, yet concise and complete. The Implications of Recent Developments in Artificial Intelligence (AI) have changed the way people think about AI. The purpose of this article is to provide some background information on AI and to provide the reader with some information about the impact of recent developments in AI.\n\nWhat is Artificial Intelligence?\n\nA computer is a computer that is capable of reasoning. A computer is not a machine. A computer is a machine with a human mind. A computer is a computer that can do any task.\n\nA computer is not a computer. A computer is a computer that can do any task. A computer is not a computer. A computer is a computer that is capable of performing any task.\n\nA computer is not a computer. A computer is a computer that is capab...",True,300,0.7,4.556854


##Comparison of texts generated by different prompts
*Text 1: The base prompt generated this text*

> The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence.\n\nWe have reached a point where we are seeing significant advances in artificial intelligence. There is an enormous amount of progress in the field of artificial intelligence, from robotics to speech recognition. But what has surprised me the most is the amount of progress that has been made in machine learning. This is a field that has been largely ignored by most researchers because it is considered a field of advanced technology.\n\nIn the past, most research in machine learning was focused on using large-scale data sets. In many ways, this approach was flawed, since the large-scale data sets were very small, and they were limited to certain tasks. In this post, I will discuss how machine learning has been used to solve many of the problems in education and the job market, and why this approach is so useful.\n\nIn the past, large-scale data se...

*Text 2: The revised prompt created this text*

> The following is a short article written for a technology magazine. Create text so that it looks like, \n it was written by a human and looks credible, include references to concrete facts, ensuring it is rich and informative, \n yet concise and complete. The Implications of Recent Developments in Artificial Intelligence\n\nI. Introduction\n\nThis article is about the recent developments in artificial intelligence (AI). I'm writing this article for a technology magazine because it is not clear that AI is the next big thing. In fact, I believe that it is already behind the next big thing. AI is not yet mature enough to be a disruptive technology in its own right.\n\nI think it is important to understand what AI is and why it is a big deal.\n\nIn order to understand the difference between AI and robotics, I need to start by explaining what robotics is. In robotics, an object is a set of movements that are performed by a robot to accomplish certain tasks....

**Observations**

The observations are based on the above texts.  Subsequent runs will generate different text.

- Text 1:
    - The format of the 1st text is like an informal blog post / article.  It is human like (though the prompt did not specify this)
    - It's more general in nature/tone
    - Reference is made to data sets were very small (though the prompt did not include a reference to facts)

- Text 2:
    - The revised prompt included more detail, as specified in the assigment header. The generated text includes this prompt.  Ignoring this, the text is human like.
    - The text includes a section heading (introduction), which you might expect in an article
    - No mention of facts.


**Conclusion**

- The prompt provided does impact on the quality of the text generated.  It may be that this model, has not been trained to deal with instructions as other models may have been, e.g. the Qwen Instruct model, as the text generated with the revised prompt did not take on any of its instructions and included the prompt in the output.
- Due to the above, I feel that text 1, is a better and more readable text.


# Reflection
It was interesting to see:
- how the diversity and coherence of the texts could be manipulated using the model parameters.
- some level of diversity is required to avoid bland text
- relative cost in generating a small amount of text - up to 6 seconds for a 300 word piece of text.
- understand the giveaways of generated text (repetition, vague / general text)
- the importance of a good prompt.